In [1]:
!pip install opencv-python opencv-contrib-python numpy matplotlib

In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from typing import List

In [3]:
IMAGE_FOLDER = "images/panorama_set"

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

RESIZE_WIDTH = 800   # helps if phone photos are very large

In [4]:
def load_images(folder):

    images = []

    for file in sorted(os.listdir(folder)):

        if Path(file).suffix.lower() in IMAGE_EXTENSIONS:

            path = os.path.join(folder, file)

            img = cv2.imread(path)

            if img is None:
                continue

            # resize for speed
            h, w = img.shape[:2]
            scale = RESIZE_WIDTH / w
            img = cv2.resize(img, (RESIZE_WIDTH, int(h*scale)))

            images.append(img)

            print("Loaded:", file, img.shape)

    return images


def show_images(images, titles=None):

    plt.figure(figsize=(15,6))

    for i,img in enumerate(images):

        plt.subplot(1,len(images),i+1)

        plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))

        if titles:
            plt.title(titles[i])

        plt.axis("off")

    plt.show()

In [5]:
def visualize_overlap(img1,img2):

    gray1=cv2.cvtColor(img1,cv2.COLOR_BGR2GRAY)
    gray2=cv2.cvtColor(img2,cv2.COLOR_BGR2GRAY)

    orb=cv2.ORB_create(2000)

    kp1,des1=orb.detectAndCompute(gray1,None)
    kp2,des2=orb.detectAndCompute(gray2,None)

    if des1 is None or des2 is None:
        print("Descriptors missing")
        return

    bf=cv2.BFMatcher(cv2.NORM_HAMMING)

    matches=bf.match(des1,des2)

    matches=sorted(matches,key=lambda x:x.distance)[:50]

    match_img=cv2.drawMatches(img1,kp1,img2,kp2,matches,None,flags=2)

    plt.figure(figsize=(12,6))
    plt.imshow(cv2.cvtColor(match_img,cv2.COLOR_BGR2RGB))
    plt.title("Feature Overlap")
    plt.axis("off")

In [6]:
def detect_and_match_features(img1,img2):

    gray1=cv2.cvtColor(img1,cv2.COLOR_BGR2GRAY)
    gray2=cv2.cvtColor(img2,cv2.COLOR_BGR2GRAY)

    orb=cv2.ORB_create(2000)

    kp1,des1=orb.detectAndCompute(gray1,None)
    kp2,des2=orb.detectAndCompute(gray2,None)

    if des1 is None or des2 is None:
        return None,None,None,None,[]

    bf=cv2.BFMatcher(cv2.NORM_HAMMING)

    matches=bf.knnMatch(des1,des2,k=2)

    good=[]

    for m,n in matches:

        if m.distance < 0.75*n.distance:
            good.append(m)

    return kp1,des1,kp2,des2,good

In [7]:
def compute_homography(kp1,kp2,good_matches):

    if len(good_matches) < 10:
        return None

    src_pts=np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1,1,2)

    dst_pts=np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1,1,2)

    H,_=cv2.findHomography(src_pts,dst_pts,cv2.RANSAC,5.0)

    return H

In [8]:
def stitch_two_images(img1,img2,H):

    h1,w1=img1.shape[:2]
    h2,w2=img2.shape[:2]

    corners=np.float32([[0,0],[0,h2],[w2,h2],[w2,0]]).reshape(-1,1,2)

    warped_corners=cv2.perspectiveTransform(corners,H)

    all_corners=np.concatenate((

        np.float32([[0,0],[0,h1],[w1,h1],[w1,0]]).reshape(-1,1,2),
        warped_corners

    ),axis=0)

    xmin,ymin=np.int32(all_corners.min(axis=0).ravel()-0.5)
    xmax,ymax=np.int32(all_corners.max(axis=0).ravel()+0.5)

    t=np.array([[1,0,-xmin],[0,1,-ymin],[0,0,1]])

    result=cv2.warpPerspective(img2,t.dot(H),(xmax-xmin,ymax-ymin))

    result[-ymin:h1-ymin,-xmin:w1-xmin]=img1

    return result

In [9]:
def simple_stitch_all(images):

    panorama=images[0].copy()

    for i in range(1,len(images)):

        print("Stitching image",i)

        kp1,des1,kp2,des2,good=detect_and_match_features(panorama,images[i])

        if len(good)<10:
            print("Not enough matches")
            continue

        H=compute_homography(kp1,kp2,good)

        if H is None:
            print("Homography failed")
            continue

        panorama=stitch_two_images(panorama,images[i],H)

    return panorama

In [10]:
images=load_images(IMAGE_FOLDER)

if len(images)<2:
    raise ValueError("Need at least two images")

show_images(images[:4])

FileNotFoundError: [Errno 2] No such file or directory: 'images/panorama_set'

In [ ]:
for i in range(len(images)-1):

    print("Checking overlap:",i,"and",i+1)

    visualize_overlap(images[i],images[i+1])

In [ ]:
manual_panorama=simple_stitch_all(images)

plt.figure(figsize=(15,8))

plt.imshow(cv2.cvtColor(manual_panorama,cv2.COLOR_BGR2RGB))

plt.title("Manual Panorama")

plt.axis("off")

plt.show()

In [ ]:
print("Running OpenCV Stitcher")

stitcher=cv2.Stitcher_create(cv2.Stitcher_PANORAMA)

status,pano=stitcher.stitch(images)

if status==cv2.Stitcher_OK:

    plt.figure(figsize=(15,8))

    plt.imshow(cv2.cvtColor(pano,cv2.COLOR_BGR2RGB))

    plt.title("OpenCV Panorama")

    plt.axis("off")

    plt.show()

else:
    print("Stitching failed:",status)

In [ ]:
stitcher_scans=cv2.Stitcher_create(cv2.Stitcher_SCANS)

status2,pano2=stitcher_scans.stitch(images)

if status2==cv2.Stitcher_OK:

    plt.figure(figsize=(15,7))

    plt.subplot(1,2,1)
    plt.imshow(cv2.cvtColor(pano,cv2.COLOR_BGR2RGB))
    plt.title("PANORAMA")

    plt.subplot(1,2,2)
    plt.imshow(cv2.cvtColor(pano2,cv2.COLOR_BGR2RGB))
    plt.title("SCANS")

    plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')